In [ ]:
from pathlib import Path

import gpxpy
import joblib
import pandas as pd
import plotly.express as px

from client import Client

CLIENT = Client()


def get_gpx_df(aid, i):
    gpx = gpxpy.parse(CLIENT.get_activity(activity_id=aid, format="gpx").decode())
    cur = pd.DataFrame(
        [
            {"lat": p.latitude, "lon": p.longitude}
            for p in gpx.tracks[0].segments[0].points
        ]
    )
    cur["route"] = f"route{i}"
    return cur

In [ ]:
res = pd.DataFrame(CLIENT.get_activities(start=0, limit=10_000))
res["activityType"] = res["activityType"].apply(lambda d: d.get("typeKey"))
res["startTimeLocal"] = pd.to_datetime(res["startTimeLocal"])

In [ ]:
df = pd.DataFrame()
aids = res.query(
    '(activityType == "running" | activityType == "trail_running") & startTimeLocal.dt.year == 2026'
)["activityId"]
print(len(aids))
df = pd.concat(
    joblib.Parallel(n_jobs=8)(
        joblib.delayed(get_gpx_df)(aid, i) for i, aid in enumerate(aids)
    )
)
print(len(df))

In [ ]:
fig = px.line_map(
    df,
    lat="lat",
    lon="lon",
    map_style="dark",
    line_group="route",
    # color="route",
    width=800,
    height=1000,
    zoom=12,
)
fig.update_traces(line={"color": "red", "width": 1})